# 3.4 · Caching and subscriptions: keeping lists fresh

**缓存与订阅：让列表保持新鲜**

> **After this lesson you can:** Set and honour ttlMs/cacheScope hints, build a correct client cache, and use subscriptions/listen to receive list-changed and resource-updated notifications.

Official reading: [Spec: Caching](https://modelcontextprotocol.io/specification/2026-07-28/server/utilities/caching) · [Spec: Subscriptions](https://modelcontextprotocol.io/specification/2026-07-28/basic/patterns/subscriptions)

Run the cells top to bottom (`Shift+Enter`), then change things and re-run. Everything runs locally: servers live in this kernel, in temp folders, or on localhost. Cells that call a real LLM are skipped unless `ANTHROPIC_API_KEY` is set.

### 1. Every cacheable result carries hints
By default `MCPServer` marks results `ttlMs: 0` (immediately stale) and `cacheScope: "private"` — the safest choice. Check every cacheable method, including the `server/discover` result the client fetched while connecting, and confirm `tools/call` carries no hints.

In [ ]:
import json
import logging
from collections import Counter

import mcp_types as types
from mcp import Client
from mcp.server import MCPServer
from mcp.server.mcpserver import Context

logging.getLogger("mcp").setLevel(logging.CRITICAL)

plain = MCPServer("plain")


@plain.tool()
def ping_tool() -> str:
    """Return pong."""
    return "pong"


@plain.resource("config://app")
def config() -> str:
    return "theme=dark"


@plain.prompt()
def hello() -> str:
    """Say hello."""
    return "Hello!"


async with Client(plain) as client:
    discover = client.session.discover_result
    rows = [("server/discover", discover), ("tools/list", await client.list_tools()),
            ("prompts/list", await client.list_prompts()), ("resources/list", await client.list_resources()),
            ("resources/templates/list", await client.list_resource_templates()),
            ("resources/read", await client.read_resource("config://app")),
            ("tools/call", await client.call_tool("ping_tool", {}))]
    for method, result in rows:
        wire = result.model_dump(by_alias=True, mode="json", exclude_none=True)
        print(f"{method:<26} ttlMs={wire.get('ttlMs', '—')!s:<7} cacheScope={wire.get('cacheScope', '—')}")

### 2. Server side: choosing hints
`MCPServer(cache_hints={method: CacheHint(ttl_ms, scope)})` sets hints per method. A middleware counts how many requests actually reach the server, so we can see the client cache work.

In [ ]:
from mcp.server.caching import CacheHint

hits = Counter()


async def count_requests(ctx, call_next):
    hits[ctx.method] += 1
    return await call_next(ctx)


server = MCPServer("travel", middleware=[count_requests], cache_hints={
    "tools/list": CacheHint(ttl_ms=300_000, scope="public"),       # same for every user, 5 min
    "resources/read": CacheHint(ttl_ms=60_000, scope="private"),    # user data, 1 min
})
inbox = {"messages": 1}


@server.tool()
def search_flights(origin: str, destination: str) -> list[str]:
    """Search flights between two airports."""
    return [f"{origin}->{destination} 08:15", f"{origin}->{destination} 17:40"]


@server.resource("inbox://me", mime_type="application/json")
def my_inbox() -> str:
    return json.dumps(inbox)


async with Client(server) as client:
    t = await client.list_tools()
    r = await client.read_resource("inbox://me")
    print("tools/list    ", t.ttl_ms, t.cache_scope)
    print("resources/read", r.ttl_ms, r.cache_scope)

### 3. Client side: the response cache and `cache_mode`
The SDK `Client` honours hints by default (`cache=CacheConfig()`). Each cacheable call takes `cache_mode`: `"use"` (serve if fresh, else fetch and store), `"refresh"` (always fetch, then store), `"bypass"` (ignore the cache). `Client(..., cache=None)` disables it.

In [ ]:
async with Client(server) as client:
    hits.clear()
    await client.list_tools()                         # miss -> server
    await client.list_tools()                         # fresh -> served from cache
    await client.list_tools(cache_mode="refresh")     # forced fetch, re-stored
    await client.list_tools(cache_mode="bypass")      # straight to server
    print("with cache:   ", dict(hits))

async with Client(server, cache=None) as client:
    hits.clear()
    await client.list_tools()
    await client.list_tools()
    print("cache=None:   ", {k: v for k, v in hits.items() if k != "server/discover"})

### 4. Freshness is measured from receive time
`CacheConfig(clock=...)` lets us fake the wall clock: the entry is fresh while `now < t_received + ttlMs`, then stale.

In [ ]:
from mcp.client.caching import CacheConfig

fake_now = [1_000.0]  # seconds

async with Client(server, cache=CacheConfig(clock=lambda: fake_now[0])) as client:
    hits.clear()
    for advance_s in [0, 120, 179.9, 0.2]:
        fake_now[0] += advance_s
        before = hits["tools/list"]
        await client.list_tools()
        print(f"t+{fake_now[0] - 1_000:6.1f}s  ->  {'server' if hits['tools/list'] > before else 'cache '}"
              f"   (ttl {300}s)")

The entry received at t=0 is fresh at 120 s and at 299.9 s (`now < 0 + 300 s`) and stale at 300.1 s, so that request goes to the server and the new response starts a fresh 300 s clock. Freshness is checked **lazily**, at the moment the data is needed: nothing was re-fetched in the background while the clock ran.

### 5. TTL is a hint, not a guarantee — notifications invalidate
A tool changes the inbox. Without a subscription the client keeps serving the cached (now wrong) value until the TTL runs out. With `subscriptions/listen`, the `notifications/resources/updated` evicts the entry immediately and the next read goes to the server.

In [ ]:
@server.tool()
async def receive_mail(ctx: Context) -> int:
    """Simulate new mail arriving."""
    inbox["messages"] += 1
    await ctx.notify_resource_updated("inbox://me")
    return inbox["messages"]


async with Client(server) as client:
    print("read:", (await client.read_resource("inbox://me")).contents[0].text)
    await client.call_tool("receive_mail", {})
    print("after change, no subscription:", (await client.read_resource("inbox://me")).contents[0].text, "<- stale!")

    async with client.listen(resource_subscriptions=["inbox://me"]) as sub:
        await client.call_tool("receive_mail", {})
        event = await anext(sub)
        hits.clear()
        print("event:", event)
        print("after notification:", (await client.read_resource("inbox://me")).contents[0].text,
              "| reached server:", hits["resources/read"] == 1)

### 6. `subscriptions/listen` on the wire
Below the SDK, over stdio: two **concurrent** subscriptions with different filters, the ack that comes first, `subscriptionId` tagging, opt-in filtering, client cancellation with `notifications/cancelled`, and a server-initiated **graceful closure**. (The `shutdown_streams` tool reaches into an SDK internal just to trigger what a server does on shutdown.)

In [ ]:
import queue
import subprocess
import sys
import tempfile
import textwrap
import threading
import time
from pathlib import Path

workdir = Path(tempfile.mkdtemp())
(workdir / "watch_server.py").write_text(textwrap.dedent('''
    from mcp.server import MCPServer
    from mcp.server.caching import CacheHint
    from mcp.server.mcpserver import Context

    server = MCPServer("watchable", cache_hints={"tools/list": CacheHint(ttl_ms=300_000, scope="public")})
    config = {"theme": "dark"}

    @server.resource("config://app", mime_type="application/json")
    def app_config() -> str:
        return str(config)

    @server.tool()
    async def set_theme(theme: str, ctx: Context) -> str:
        """Change the theme (updates config://app)."""
        config["theme"] = theme
        await ctx.notify_resource_updated("config://app")
        return "ok"

    @server.tool()
    async def install_plugin(name: str, ctx: Context) -> str:
        """Install a plugin, which adds a tool."""
        server.add_tool(lambda: f"{name} says hi", name=f"{name}_hello", description=f"Greeting from {name}")
        await ctx.notify_tools_changed()
        return "installed"

    @server.tool()
    async def shutdown_streams() -> str:
        """Demo only: end all listen streams gracefully, as on server shutdown."""
        server._lowlevel_server._request_handlers["subscriptions/listen"].handler.close()
        return "closing"

    if __name__ == "__main__":
        server.run(transport="stdio")
'''))

proc = subprocess.Popen([sys.executable, str(workdir / "watch_server.py")], stdin=subprocess.PIPE,
                        stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
inbound: queue.Queue = queue.Queue()
threading.Thread(target=lambda: [inbound.put(line) for line in proc.stdout], daemon=True).start()
META = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
        "io.modelcontextprotocol/clientCapabilities": {},
        "io.modelcontextprotocol/clientInfo": {"name": "by-hand", "version": "0.1"}}


def send(message):
    message.setdefault("params", {})["_meta"] = META
    shown = {k: v for k, v in message.items() if k != "jsonrpc"}
    shown["params"] = {k: v for k, v in message["params"].items() if k != "_meta"}
    print(">>>", json.dumps(shown))
    proc.stdin.write(json.dumps(message) + "\n")
    proc.stdin.flush()


def drain(seconds=0.6):
    deadline = time.time() + seconds
    while (left := deadline - time.time()) > 0:
        try:
            msg = json.loads(inbound.get(timeout=left))
        except queue.Empty:
            break
        if "result" in msg and "tools" in msg["result"]:
            msg["result"] = {k: msg["result"][k] for k in ("ttlMs", "cacheScope")} | {"tools": "[...]"}
        elif "result" in msg:
            msg["result"].get("_meta", {}).pop("io.modelcontextprotocol/serverInfo", None)
            msg["result"] = {k: v for k, v in msg["result"].items()
                             if k in ("resultType", "structuredContent") or (k == "_meta" and v)}
        print("    <<<", json.dumps({k: v for k, v in msg.items() if k != "jsonrpc"}))


send({"jsonrpc": "2.0", "id": 1, "method": "tools/list"}); drain()
send({"jsonrpc": "2.0", "id": "tools-sub", "method": "subscriptions/listen",
      "params": {"notifications": {"toolsListChanged": True}}})
send({"jsonrpc": "2.0", "id": "cfg-sub", "method": "subscriptions/listen",
      "params": {"notifications": {"resourceSubscriptions": ["config://app"]}}}); drain()
print("\n# a resource change reaches only cfg-sub:")
send({"jsonrpc": "2.0", "id": 2, "method": "tools/call", "params": {"name": "set_theme", "arguments": {"theme": "light"}}}); drain()
print("\n# a tool-list change reaches only tools-sub:")
send({"jsonrpc": "2.0", "id": 3, "method": "tools/call", "params": {"name": "install_plugin", "arguments": {"name": "maps"}}}); drain()
print("\n# cancel cfg-sub, then change the resource again: silence")
send({"jsonrpc": "2.0", "method": "notifications/cancelled", "params": {"requestId": "cfg-sub"}}); drain(0.3)
send({"jsonrpc": "2.0", "id": 4, "method": "tools/call", "params": {"name": "set_theme", "arguments": {"theme": "solarized"}}}); drain()
print("\n# the server shuts its streams down gracefully:")
send({"jsonrpc": "2.0", "id": 5, "method": "tools/call", "params": {"name": "shutdown_streams", "arguments": {}}}); drain()
proc.stdin.close()
_ = proc.wait(timeout=10)

Read it closely: each ack precedes any notification on its subscription; every stream message carries the listen request's id as `subscriptionId`; the final message is a **response** with id `tools-sub` — the graceful closure.

### 7. The same with the SDK: `client.listen()`
`client.listen(...)` sends the request, waits for the ack (`sub.honored`, `sub.subscription_id`), and yields typed events. Two subscriptions can run at once; when the server closes gracefully the `async for` loop simply ends.

In [ ]:
catalogue = MCPServer("catalogue")
prompts_enabled = []


@catalogue.tool()
async def add_feature(name: str, ctx: Context) -> str:
    """Add a prompt and a tool, then tell subscribers."""
    catalogue.add_tool(lambda: name, name=f"{name}_tool", description=f"{name} tool")
    await ctx.notify_tools_changed()
    prompts_enabled.append(name)
    await ctx.notify_prompts_changed()
    return "added"


async with Client(catalogue) as client:
    async with client.listen(tools_list_changed=True) as tools_sub, \
               client.listen(prompts_list_changed=True, resources_list_changed=True) as other_sub:
        print("subscription ids:", tools_sub.subscription_id, other_sub.subscription_id)
        print("honoured:", tools_sub.honored.model_dump(by_alias=True, exclude_none=True),
              other_sub.honored.model_dump(by_alias=True, exclude_none=True))
        await client.call_tool("add_feature", {"name": "maps"})
        print(f"{tools_sub.subscription_id} got:", await anext(tools_sub))
        print(f"{other_sub.subscription_id} got:", await anext(other_sub))

        # graceful closure from the server side (SDK internal, demo only)
        catalogue._lowlevel_server._request_handlers["subscriptions/listen"].handler.close()
        leftovers = [event async for event in tools_sub]
        print("tools_sub ended gracefully; leftover events:", leftovers)

### 8. Break it: a mislabelled `public` result in a shared cache
A gateway shares one cache store between users, partitioned by authorization context, and opts in to sharing `public` entries (`share_public=True`). If the server labels a *private* inbox read as `public`, Bob is served Alice's mail without the server ever seeing his request.

In [ ]:
from mcp.client.caching import InMemoryResponseCacheStore

mail = {"alice": "Alice: your flight is confirmed", "bob": "Bob: nothing new"}


def make_mail_server(scope):
    srv = MCPServer("mail", middleware=[count_requests],
                    cache_hints={"resources/read": CacheHint(ttl_ms=60_000, scope=scope)})
    current_user = {"name": "alice"}   # stand-in for the authenticated principal

    @srv.resource("mail://inbox")
    def mailbox() -> str:
        return mail[current_user["name"]]

    return srv, current_user


for scope in ["public", "private"]:
    srv, current_user = make_mail_server(scope)
    store = InMemoryResponseCacheStore()   # one shared gateway cache
    hits.clear()
    for user in ["alice", "bob"]:
        current_user["name"] = user
        config = CacheConfig(store=store, partition=user, target_id="mail-server", share_public=True)
        async with Client(srv, cache=config) as client:
            text = (await client.read_resource("mail://inbox")).contents[0].text
            print(f"cacheScope={scope:<7} {user:<5} sees: {text!r}")
    print(f"   server saw {hits['resources/read']} resources/read request(s)\n")

With `private`, each user's entry lives in its own partition and Bob's read goes to the server. The fix belongs on the **server** (label user data `private`, enforce access on every request) — and gateways should only enable `share_public` for servers they trust.

### Your turn
1. Give `prompts/list` on `server` a 10-minute public TTL and verify with the middleware counter that a second `list_prompts()` never reaches the server. Then add a prompt from a tool with `ctx.notify_prompts_changed()` and show the cache is invalidated only when the client is listening with `prompts_list_changed=True`.
2. In section 6, open a third subscription that asks for `promptsListChanged` only, run `install_plugin` again, and confirm it receives nothing. Which spec rule guarantees this?
3. In section 6, send `subscriptions/listen` with an empty `notifications` object. What does the ack contain, and what would a careful client do with it?
4. Write a `fetch_tools(client)` helper for a host that serves the cached list when fresh, re-fetches with `cache_mode="refresh"` after any `ToolsListChanged` event, and falls back to the stale copy if the re-fetch raises.